# 🎬 VIDEO QUIZ GENERATOR STUDIO — GOOGLE COLAB RUNNER
Notebook này cho phép bạn khởi chạy toàn bộ Studio (Backend + Frontend Remotion) trực tiếp trên Google Colab có GPU/CPU miễn phí.
Chỉ gồm **3 bước (3 Cells)** đơn giản dưới đây:

In [ ]:
# ==============================================================================
# CELL 1: CLONE REPOSITORY TỪ GITHUB (HỖ TRỢ PRIVATE REPO QUA TOKEN)
# ==============================================================================
import os
import shutil

# --- [BƯỚC 1]: ĐIỀN THÔNG TIN REPOSITORY VÀ GITHUB TOKEN CỦA BẠN DƯỚI ĐÂY ---
# Ví dụ: GITHUB_REPO = "username/video-quiz-generator"
GITHUB_REPO = "YOUR_USERNAME/YOUR_REPOSITORY"  # <-- Điền username/repo tại đây

# Tạo token tại: GitHub -> Settings -> Developer settings -> Personal access tokens (classic)
# Quyền (scope) cần thiết: 'repo' (Full control of private repositories)
GITHUB_TOKEN = "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN"  # <-- Điền Personal Access Token tại đây

# ------------------------------------------------------------------------------
REPO_NAME = GITHUB_REPO.split('/')[-1].replace('.git', '') if '/' in GITHUB_REPO else "Video-quiz-new"
WORKSPACE_DIR = f"/content/{REPO_NAME}"

print(f"[*] Đang chuẩn bị clone repository: {GITHUB_REPO}")

if os.path.exists(WORKSPACE_DIR) and os.path.exists(os.path.join(WORKSPACE_DIR, 'package.json')):
    print(f"[✓] Thư mục dự án đã tồn tại tại {WORKSPACE_DIR}. Tiến hành kéo cập nhật mới nhất (git pull)...")
    %cd {WORKSPACE_DIR}
    !git pull origin main || !git pull origin master
else:
    %cd /content
    if GITHUB_TOKEN and GITHUB_TOKEN != "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN":
        AUTH_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    else:
        AUTH_URL = f"https://github.com/{GITHUB_REPO}.git"
    
    !git clone {AUTH_URL} {REPO_NAME}
    %cd {WORKSPACE_DIR}

print(f"\n[✓] Đã chuyển vào thư mục làm việc: {os.getcwd()}")


In [ ]:
# ==============================================================================
# CELL 2: KIỂM TRA & CÀI ĐẶT DEPENDENCY THÔNG MINH (SKIP NẾU ĐÃ CÓ / CÓ CACHE)
# ==============================================================================
import subprocess
import shutil
import os
import sys

def check_command(cmd_name):
    """Kiểm tra một lệnh CLI hệ thống đã tồn tại trong PATH hay chưa."""
    return shutil.which(cmd_name) is not None

def check_python_package(package_name):
    """Kiểm tra một Python package đã được cài đặt hay chưa."""
    try:
        __import__(package_name.replace('-', '_'))
        return True
    except ImportError:
        return False

print("=== KIỂM TRA MÔI TRƯỜNG & DEPENDENCIES ===")

# 1. Kiểm tra Node.js (Yêu cầu Node >= 18 cho Remotion & Vite)
node_ok = False
if check_command("node"):
    try:
        node_ver = subprocess.check_output(["node", "-v"]).decode().strip()
        major = int(node_ver.lstrip('v').split('.')[0])
        if major >= 18:
            print(f"[✓] Node.js đã sẵn sàng: {node_ver}")
            node_ok = True
    except Exception:
        pass

if not node_ok:
    print("[*] Đang cài đặt Node.js v20 LTS...")
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
    !apt-get install -y nodejs > /dev/null 2>&1
    print("[✓] Đã cài đặt Node.js thành công!")

# 2. Kiểm tra FFmpeg
if check_command("ffmpeg"):
    ffmpeg_ver = subprocess.check_output(["ffmpeg", "-version"]).decode().split('\n')[0]
    print(f"[✓] FFmpeg đã sẵn sàng: {ffmpeg_ver}")
else:
    print("[*] Đang cài đặt FFmpeg...")
    !apt-get update -qq && !apt-get install -y -qq ffmpeg > /dev/null 2>&1
    print("[✓] Đã cài đặt FFmpeg thành công.")

# 3. Kiểm tra Python packages (edge-tts)
python_pkgs = ["edge-tts"]
for pkg in python_pkgs:
    if check_python_package(pkg):
        print(f"[✓] Python package '{pkg}' đã sẵn sàng.")
    else:
        print(f"[*] Đang cài đặt Python package '{pkg}'...")
        !pip install -q {pkg}
        print(f"[✓] Đã cài đặt '{pkg}'.")

# 4. Kiểm tra & Cài đặt Node modules (Tận dụng Cache)
if os.path.exists("node_modules") and os.path.exists("node_modules/remotion"):
    print("[✓] Thư mục node_modules đã tồn tại. Bỏ qua npm install để tiết kiệm thời gian.")
else:
    print("[*] Đang cài đặt npm packages (sử dụng cache)... Chờ khoảng 1-2 phút...")
    !npm install --prefer-offline --no-audit --loglevel=error
    print("[✓] npm install hoàn tất!")


In [ ]:
# ==============================================================================
# CELL 3: KHỞI CHẠY STUDIO & HỆ THỐNG ĐA ĐƯỜNG HẦM (COLAB DIRECT + CLOUDFLARE + LOCALTUNNEL)
# ==============================================================================
import subprocess
import time
import urllib.request
import re
import os
import shutil
from IPython.display import display, HTML

# Cấu hình Port theo dự án
FRONTEND_PORT = 5400
BACKEND_PORT = 5410

print("=== KHỞI CHẠY BACKEND & FRONTEND ===")

# Dọn dẹp process cũ nếu có
!fuser -k 5400/tcp > /dev/null 2>&1 || true
!fuser -k 5410/tcp > /dev/null 2>&1 || true
!pkill -f cloudflared > /dev/null 2>&1 || true
!pkill -f localtunnel > /dev/null 2>&1 || true

# Khởi động Backend & Frontend ở chế độ background
log_file = open("/content/studio_server.log", "w")
server_proc = subprocess.Popen(
    ["npm", "run", "dev"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    shell=False
)

print("[*] Đang chờ Studio Video khởi động hoàn tất...")
server_ready = False
for attempt in range(40):
    time.sleep(2)
    try:
        with urllib.request.urlopen(f"http://localhost:{FRONTEND_PORT}", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        pass
    print(f"    ... đang kết nối server (lần {attempt + 1}/40)")

if not server_ready:
    print("[!] CẢNH BÁO: Server chưa phản hồi kịp hoặc gặp lỗi. Xem log dưới đây:")
    !tail -n 25 /content/studio_server.log
else:
    print(f"[✓] Studio Video đã sẵn sàng trên cổng {FRONTEND_PORT}!\n")

# ------------------------------------------------------------------------------
# 1. ĐƯỜNG LINK 1 (NATIVE): Google Colab Direct Proxy (100% Ổn định, không qua trung gian)
# ------------------------------------------------------------------------------
colab_direct_url = None
try:
    from google.colab.output import eval_js
    colab_direct_url = eval_js(f"google.colab.kernel.proxyPort({FRONTEND_PORT})")
    print(f"[✓] Đã tạo Google Colab Direct Proxy: {colab_direct_url}")
except Exception as e:
    print(f"[-] Chưa khởi tạo được Google Colab Direct: {e}")

# ------------------------------------------------------------------------------
# 2. ĐƯỜNG LINK 2 (PUBLIC TUNNEL): Thử Cloudflare, nếu timeout thì tự động chuyển Localtunnel
# ------------------------------------------------------------------------------
def extract_cf_url(log_path):
    if not os.path.exists(log_path):
        return None
    try:
        with open(log_path, "r", errors="ignore") as f:
            content = f.read()
        matches = re.findall(r"https://([a-zA-Z0-9\-]+)\.trycloudflare\.com", content)
        valid = [f"https://{m}.trycloudflare.com" for m in matches if m.lower() != "api"]
        if valid:
            return valid[-1]
    except Exception:
        pass
    return None

public_url = None
tunnel_type = "Cloudflare"
tunnel_proc = None
tunnel_pass = None

# Cài đặt cloudflared nếu chưa có
if not shutil.which("cloudflared"):
    print("[*] Đang tải công cụ Cloudflare Tunnel...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("[*] Đang thử kết nối Cloudflare Quick Tunnel...")
cf_log = "/content/cloudflared.log"
!rm -f {cf_log}
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{FRONTEND_PORT}"],
    stdout=open(cf_log, "w"),
    stderr=subprocess.STDOUT
)

# Chờ Cloudflare tối đa 15 giây
for _ in range(12):
    time.sleep(1.5)
    public_url = extract_cf_url(cf_log)
    if public_url:
        break

# Nếu Cloudflare bị timeout (api.trycloudflare.com context deadline exceeded)
if not public_url:
    print("[!] Cloudflare Tunnel đang bị quá tải hoặc Timeout API từ Cloudflare.")
    print("[*] Đang tự động chuyển sang đường hầm dự phòng: LOCALTUNNEL...")
    try:
        tunnel_proc.terminate()
    except Exception:
        pass
    
    # Lấy IP Public của Colab làm Password cho Localtunnel
    try:
        tunnel_pass = urllib.request.urlopen("https://ipv4.icanhazip.com", timeout=4).read().decode().strip()
    except Exception:
        tunnel_pass = ""

    lt_log = "/content/localtunnel.log"
    !rm -f {lt_log}
    tunnel_proc = subprocess.Popen(
        ["npx", "--yes", "localtunnel", "--port", str(FRONTEND_PORT)],
        stdout=open(lt_log, "w"),
        stderr=subprocess.STDOUT
    )
    tunnel_type = "Localtunnel"
    for _ in range(20):
        time.sleep(1.5)
        if os.path.exists(lt_log):
            with open(lt_log, "r", errors="ignore") as f:
                c = f.read()
                m = re.search(r"https://[a-zA-Z0-9\-]+\.loca\.lt", c)
                if m:
                    public_url = m.group(0)
                    break

print("\n" + "=" * 65)
print("🚀 STUDIO VIDEO QUIZ ĐÃ SẴN SÀNG TRÊN GOOGLE COLAB!")
print("=" * 65)

html_buttons = ""
if colab_direct_url:
    print(f"\n👉 [LINK 1 - KHUYÊN DÙNG] GOOGLE COLAB NATIVE PROXY:\n   {colab_direct_url}\n   (Mở ngay trong tab mới, 100% ổn định, không lo lỗi timeout tunnel)")
    html_buttons += f'''
        <a href="{colab_direct_url}" target="_blank" style="display:inline-block;background:#10b981;color:#ffffff;font-weight:bold;font-size:15px;padding:12px 24px;border-radius:8px;text-decoration:none;margin:8px;box-shadow:0 0 12px rgba(16,185,129,0.4);">
            👉 LINK 1: MỞ QUA GOOGLE COLAB PROXY (Khuyên Dùng)
        </a>
    '''

if public_url:
    pass_note = f" (Mật khẩu IP: {tunnel_pass})" if tunnel_type == "Localtunnel" and tunnel_pass else ""
    print(f"\n👉 [LINK 2] PUBLIC INTERNET TUNNEL ({tunnel_type}):\n   {public_url}{pass_note}\n")
    html_buttons += f'''
        <a href="{public_url}" target="_blank" style="display:inline-block;background:#00e5ff;color:#000000;font-weight:bold;font-size:15px;padding:12px 24px;border-radius:8px;text-decoration:none;margin:8px;box-shadow:0 0 12px rgba(0,229,255,0.4);">
            🌐 LINK 2: MỞ QUA {tunnel_type.upper()} TUNNEL
        </a>
    '''

try:
    display(HTML(f'''
        <div style="background:#0f172a;border:2px solid #38bdf8;border-radius:12px;padding:20px;text-align:center;margin:15px 0;">
            <h3 style="color:#ffffff;margin:0 0 15px 0;">🎬 Studio Video Quiz Sẵn Sàng — Bấm Link Để Mở:</h3>
            <div style="display:flex;justify-content:center;flex-wrap:wrap;">
                {html_buttons}
            </div>
            {f'<p style="color:#fbbf24;font-size:13px;margin:10px 0 0 0;">🔑 Mật khẩu IP Localtunnel (nếu web hỏi): <b>{tunnel_pass}</b></p>' if tunnel_type == "Localtunnel" and tunnel_pass else ''}
        </div>
    '''))
except Exception:
    pass

print("=" * 65)
print("[*] 🔒 DUY TRÌ KẾT NỐI: Cell này sẽ GIỮ CHẠY LIÊN TỤC để bạn Render video an toàn.")
print("[*] ⚠️ VUI LÒNG KHÔNG BẤM NÚT DỪNG (STOP) CELL NÀY KHI ĐANG SỬ DỤNG HOẶC ĐANG RENDER!")
print("=" * 65 + "\n")

# Vòng lặp duy trì vĩnh viễn (Heartbeat & Keepalive)
start_time = time.time()
last_ping = 0
try:
    while True:
        time.sleep(3)
        elapsed = int(time.time() - start_time)
        
        # Giám sát Tunnel
        if tunnel_proc and tunnel_proc.poll() is not None:
            print(f"[{time.strftime('%H:%M:%S')}] [!] Tunnel ({tunnel_type}) bị ngắt. Đang tự động kết nối lại...")
            if tunnel_type == "Cloudflare":
                tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://localhost:{FRONTEND_PORT}"], stdout=open(cf_log, "a"), stderr=subprocess.STDOUT)
            else:
                tunnel_proc = subprocess.Popen(["npx", "--yes", "localtunnel", "--port", str(FRONTEND_PORT)], stdout=open(lt_log, "a"), stderr=subprocess.STDOUT)

        # Giám sát Server
        if server_proc.poll() is not None:
            print(f"[{time.strftime('%H:%M:%S')}] [!] Cảnh báo: Server dev bị dừng đột ngột. Chi tiết log cuối:")
            with open("/content/studio_server.log", "r") as f:
                print(f.read()[-1000:])
            break

        # Heartbeat mỗi 60 giây
        if elapsed - last_ping >= 60:
            last_ping = elapsed
            mins = elapsed // 60
            active_link = colab_direct_url or public_url
            print(f"[{time.strftime('%H:%M:%S')}] [Heartbeat] Studio đang chạy ổn định ({mins} phút) | Link: {active_link}")

except KeyboardInterrupt:
    print("\n[✓] Đã nhận lệnh dừng từ người dùng. Tiến hành tắt Server và giải phóng Tunnel...")
    try:
        tunnel_proc.terminate()
    except Exception:
        pass
    try:
        server_proc.terminate()
    except Exception:
        pass
    print("[✓] Đã tắt an toàn!")
